In [0]:
# secret details

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import current_timestamp, col,count

## Customers

Ingests `olist_customers_dataset.csv`  from the raw volume into the bronze table `olist_project.bronze.customers`.

* **Grain:** one row per customer per order — `customer_id` is unique to each order, while `customer_unique_id` identifies the same shopper across multiple orders
* **Columns:** customer ID, unique customer ID, zip-code prefix, city, and state
* **Audit columns added:** `ingestion_date` (load timestamp) and `SourceFile` (raw file path)

In [0]:
cust_df = spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/olist_project/raw/raw_files/raw/olist_customers_dataset.csv')

cust_df = cust_df.withColumn('ingestion_date', current_timestamp())\
            .withColumn('SourceFile', col('_metadata.file_path'))

cust_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist_project.bronze.customers")

print(f"Customers data loaded into bronze table: {cust_df.count()}")

Customers data loaded into bronze table: 99441


## Geolocation

Ingests `olist_geolocation_dataset.csv`  from the raw volume into the bronze table `olist_project.bronze.geolocation`.

* **Grain:** one row per zip-code prefix suffix — a single prefix can map to multiple coordinate points
* **Columns:** zip-code prefix, lat/long coordinates, city, and state
* **Use:** joined downstream to enrich customers and sellers with geographic coordinates

In [0]:
geo_df = spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/olist_project/raw/raw_files/raw/olist_geolocation_dataset.csv')

geo_df = geo_df.withColumn('ingestion_date', current_timestamp())\
            .withColumn('SourceFile', col('_metadata.file_path'))

geo_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist_project.bronze.geolocation")

print(f"Geolocation data loaded into bronze table: {geo_df.count()}")

Geolocation data loaded into bronze table: 1000163


## Product Category Name Translation

Ingests `product_category_name_translation.csv` (71 categories) from the raw volume into the bronze table `olist_project.bronze.product_category_name_translation`.

* **Grain:** one row per product category
* **Columns:** category name in Portuguese and its English translation
* **Use:** lookup table to translate product categories to English in the silver/gold layers

In [0]:
product_category_df = spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/olist_project/raw/raw_files/raw/product_category_name_translation.csv')

product_category_df = product_category_df.withColumn('ingestion_date', current_timestamp())\
            .withColumn('SourceFile', col('_metadata.file_path'))

product_category_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist_project.bronze.product_category_name_translation")

print(f"Product_Category_Translation data loaded into bronze table: {product_category_df.count()}")

Product_Category_Translation data loaded into bronze table: 71


## Sellers

Ingests `olist_sellers_dataset.csv`  from the raw volume into the bronze table `olist_project.bronze.sellers`.

* **Grain:** one row per seller
* **Columns:** seller ID, zip-code prefix, city, and state
* **Use:** seller dimension, joined to order items to attribute sales and freight to each seller

In [0]:
sellers_df = spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/olist_project/raw/raw_files/raw/olist_sellers_dataset.csv')

sellers_df = sellers_df.withColumn('ingestion_date', current_timestamp())\
            .withColumn('SourceFile', col('_metadata.file_path'))

sellers_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist_project.bronze.sellers")

print(f"Sellers data loaded into bronze table: {sellers_df.count()}")

Sellers data loaded into bronze table: 3095


## Products

Ingests `olist_products_dataset.csv`  from the raw volume into the bronze table `olist_project.bronze.products`.

* **Grain:** one row per product
* **Columns:** product ID, category name (Portuguese), name length, description length, number of photos, weight in grams, and dimensions (length, height, width in cm)
* **Use:** core product dimension, with categories translated via the translation table

In [0]:
products_df = spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/olist_project/raw/raw_files/raw/olist_products_dataset.csv')

products_df = products_df.withColumn('ingestion_date', current_timestamp())\
            .withColumn('SourceFile', col('_metadata.file_path'))

products_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist_project.bronze.products")

print(f"Products data loaded into bronze table: {products_df.count()}")

Products data loaded into bronze table: 32951


## Orders

Ingests `olist_orders_dataset.csv`  from the raw volume into the bronze table `olist_project.bronze.orders`.

* **Grain:** one row per order
* **Columns:** order ID, customer ID, order status (e.g. delivered, shipped, canceled), and the full order lifecycle timestamps — purchase, approval, carrier handoff, estimated delivery, and actual delivery
* **Use:** central fact table that links customers to their order events

In [0]:
orders_df = spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/olist_project/raw/raw_files/raw/olist_orders_dataset.csv')

orders_df = orders_df.withColumn('ingestion_date', current_timestamp())\
            .withColumn('SourceFile', col('_metadata.file_path'))

orders_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist_project.bronze.orders")

print(f"Orders data loaded into bronze table: {orders_df.count()}")

Orders data loaded into bronze table: 99441


## Order Items

Ingests `olist_order_items_dataset.csv`  from the raw volume into the bronze table `olist_project.bronze.orders_items`.

* **Grain:** one row per item within an order (an order can contain several items)
* **Columns:** order ID, order item sequence number, product ID, seller ID, shipping limit date, price, and freight value
* **Use:** bridge between orders, products, and sellers — the most granular sales fact

In [0]:
orders_items_df = spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/olist_project/raw/raw_files/raw/olist_order_items_dataset.csv')

orders_items_df = orders_items_df.withColumn('ingestion_date', current_timestamp())\
            .withColumn('SourceFile', col('_metadata.file_path'))

orders_items_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist_project.bronze.orders_items")

print(f"Orders Items data loaded into bronze table: {orders_items_df.count()}")

Orders Items data loaded into bronze table: 112650


## Order Reviews

Ingests `olist_order_reviews_dataset.csv`  from the raw volume into the bronze table `olist_project.bronze.orders_reviews`.

* **Grain:** one row per review — a single order can have multiple reviews over time
* **Columns:** review ID, order ID, review score (1–5), optional review title and comment, plus creation and answer timestamps
* **Use:** customer-satisfaction signal for quality and churn analysis

In [0]:
orders_reviews_df = spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/olist_project/raw/raw_files/raw/olist_order_reviews_dataset.csv')

orders_reviews_df = orders_reviews_df.withColumn('ingestion_date', current_timestamp())\
            .withColumn('SourceFile', col('_metadata.file_path'))

orders_reviews_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist_project.bronze.orders_reviews")

print(f"Orders Reviews data loaded into bronze table: {orders_reviews_df.count()}")

Orders Reviews data loaded into bronze table: 104162


## Order Payments

Ingests `olist_order_payments_dataset.csv`  from the raw volume into the bronze table `olist_project.bronze.orders_payments`.

* **Grain:** one row per payment within an order — an order can be split across multiple payments
* **Columns:** order ID, payment sequence, payment type (credit card, boleto, voucher, debit card), installments, and payment value
* **Use:** revenue source of truth — the sum of payment values reconciles to order totals

In [0]:
orders_payments_df = spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/olist_project/raw/raw_files/raw/olist_order_payments_dataset.csv')

orders_payments_df = orders_payments_df.withColumn('ingestion_date', current_timestamp())\
            .withColumn('SourceFile', col('_metadata.file_path'))

orders_payments_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist_project.bronze.orders_payments")

print(f"Orders Payments data loaded into bronze table: {orders_payments_df.count()}")

Orders Payments data loaded into bronze table: 103886
